In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [4]:
import json
from datetime import datetime
from typing import Optional

# ✅ STEP 1: Define standalone tool functions OUTSIDE the class
def get_salesforce_deal(deal_id: str):
    """Fetch a deal from Salesforce"""
    deal_data = {
        "id": deal_id,
        "amount": 50000,
        "stage": "Closed Won"
    }
    return deal_data

def generate_draft(data_output: dict, user_input: str):
    """Generate follow-up draft"""
    return {
        "draft": f"Follow up on {data_output.get('id')}",
        "tone": "professional"
    }

def score_deal_risk(data_output: dict):
    """Score deal risk"""
    return {"risk_level": "low", "confidence": 0.95}

# ✅ STEP 2: Define the Agent class
class Agent:
    """Base agent with memory and tool registry."""
    def __init__(self, name: str, role: str, tools: dict = None):
        self.name = name
        self.role = role
        self.tools = tools or {}
        self.conversation_history = []
    
    def call_tool(self, tool_name: str, **kwargs) -> str:
        """Execute a tool and log."""
        if tool_name not in self.tools:
            return f"Error: tool {tool_name} not found"
        result = self.tools[tool_name](**kwargs)
        self.log_call(tool_name, kwargs, result)
        return result
    
    def log_call(self, tool_name: str, inputs: dict, output):
        """Observability: log all calls."""
        entry = {
            "timestamp": datetime.now().isoformat(),
            "agent": self.name,
            "tool": tool_name,
            "inputs": inputs,
            "output": output
        }
        self.conversation_history.append(entry)
        print(f"[{self.name}] {tool_name}: {json.dumps(output, indent=2)[:100]}...")

class Orchestrator(Agent):
    """Routes queries, holds state, manages retries."""
    def __init__(self, agents: dict):
        super().__init__("Orchestrator", "workflow_coordinator")
        self.agents = agents
        self.state = {"current_deal": None, "attempt_count": 0}
        self.max_attempts = 3
    
    def process(self, user_input: str) -> dict:
        """Main entry point"""
        self.state["attempt_count"] = 0
        deal_id = self._extract_deal_id(user_input)
        self.state["current_deal"] = deal_id
        
        # ✅ Use call_tool() instead of direct method call
        data_output = self.agents["data"].call_tool("get_salesforce_deal", deal_id=deal_id)
        draft = self.agents["drafter"].call_tool("generate_draft", 
                                                  data_output=data_output, 
                                                  user_input=user_input)
        risk = self.agents["risk_scorer"].call_tool("score_deal_risk", data_output=data_output)
        
        return {
            "deal_id": deal_id,
            "deal_snapshot": data_output,
            "follow_up_draft": draft,
            "risk_assessment": risk,
            "observability_log": self.conversation_history,
            "attempts": self.state["attempt_count"]
        }
    
    def _extract_deal_id(self, user_input: str) -> str:
        """Parse deal ID from user input."""
        import re
        match = re.search(r'DEAL-\d+', user_input)
        return match.group(0) if match else "DEAL-001"

# ✅ STEP 3: Initialize agents with the defined functions
data_agent = Agent("DataAgent", "pulls_salesforce_data", 
                   tools={"get_salesforce_deal": get_salesforce_deal})
drafter_agent = Agent("DrafterAgent", "writes_follow_ups", 
                      tools={"generate_draft": generate_draft})
risk_agent = Agent("RiskScorerAgent", "scores_deal_health", 
                   tools={"score_deal_risk": score_deal_risk})
evaluator_agent = Agent("EvaluatorAgent", "quality_gate")

orchestrator = Orchestrator({
    "data": data_agent,
    "drafter": drafter_agent,
    "risk_scorer": risk_agent,
    "evaluator": evaluator_agent
})

# Demo
if __name__ == "__main__":
    result = orchestrator.process("What's the status on DEAL-001? Should I follow up?")
    print(json.dumps(result, indent=2))

[DataAgent] get_salesforce_deal: {
  "id": "DEAL-001",
  "amount": 50000,
  "stage": "Closed Won"
}...
[DrafterAgent] generate_draft: {
  "draft": "Follow up on DEAL-001",
  "tone": "professional"
}...
[RiskScorerAgent] score_deal_risk: {
  "risk_level": "low",
  "confidence": 0.95
}...
{
  "deal_id": "DEAL-001",
  "deal_snapshot": {
    "id": "DEAL-001",
    "amount": 50000,
    "stage": "Closed Won"
  },
  "follow_up_draft": {
    "draft": "Follow up on DEAL-001",
    "tone": "professional"
  },
  "risk_assessment": {
    "risk_level": "low",
    "confidence": 0.95
  },
  "observability_log": [],
  "attempts": 0
}


In [1]:
# In Kaggle notebook, Cell 1:
!pip install anthropic -q

# Cell 2: Paste evaluator_agent_v2.py code
"""
Evaluator Agent: Hybrid Claude API + Deterministic Fallback Scoring
Demonstrates: tool use, state management, conditional retries, observability, multi-agent integration

Scoring dimensions:
- Deal Risk Assessment (financial, timeline, stakeholder)
- Budget Fit (alignment with deal size)
- Timeline Feasibility (realistic delivery)
- Proposal Quality (completeness, clarity, alignment)
- Recommendation (approve/reject/revise)
"""

import json
import logging
from typing import Any, Dict, Optional
from datetime import datetime
import anthropic

# ============================================================================
# LOGGING SETUP
# ============================================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - [%(name)s] - %(levelname)s - %(message)s"
)
logger = logging.getLogger("EvaluatorAgent")


# ============================================================================
# SCORING SCHEMA & VALIDATION
# ============================================================================

class ScoringResult:
    """Structured result from evaluation."""
    
    def __init__(self, 
                 deal_id: str,
                 risk_score: float,           # 0-100 (higher = more risky)
                 budget_fit_score: float,      # 0-100 (higher = better fit)
                 timeline_score: float,        # 0-100 (higher = more feasible)
                 quality_score: float,         # 0-100 (higher = better quality)
                 recommendation: str,          # "approve", "revise", "reject"
                 reasoning: str,
                 attempt: int,
                 method: str):                 # "claude_api" or "deterministic"
        
        self.deal_id = deal_id
        self.risk_score = risk_score
        self.budget_fit_score = budget_fit_score
        self.timeline_score = timeline_score
        self.quality_score = quality_score
        self.recommendation = recommendation
        self.reasoning = reasoning
        self.attempt = attempt
        self.method = method
        self.timestamp = datetime.now().isoformat()
    
    def to_dict(self) -> Dict[str, Any]:
        """Convert to dictionary for JSON serialization."""
        return {
            "deal_id": self.deal_id,
            "risk_score": self.risk_score,
            "budget_fit_score": self.budget_fit_score,
            "timeline_score": self.timeline_score,
            "quality_score": self.quality_score,
            "composite_score": (
                (100 - self.risk_score) * 0.3 +  # Lower risk is better
                self.budget_fit_score * 0.2 +
                self.timeline_score * 0.2 +
                self.quality_score * 0.3
            ),
            "recommendation": self.recommendation,
            "reasoning": self.reasoning,
            "attempt": self.attempt,
            "method": self.method,
            "timestamp": self.timestamp
        }


# ============================================================================
# DETERMINISTIC SCORING FALLBACK (Rule-Based)
# ============================================================================

def deterministic_score(deal_data: Dict[str, Any], 
                       proposal_data: Dict[str, Any],
                       attempt: int) -> ScoringResult:
    """
    Deterministic scoring based on explicit rules.
    No API calls required; fully self-contained.
    
    Rules:
    1. Risk Score: Based on contract value, timeline, unknowns
    2. Budget Fit: Proposal cost vs deal budget
    3. Timeline: Days available vs days required
    4. Quality: Completeness of proposal fields + clarity indicators
    5. Recommendation: Composite decision
    """
    deal_id = deal_data.get("deal_id", "UNKNOWN")
    
    logger.info(f"[Attempt {attempt}] Using deterministic scoring for {deal_id}")
    
    # --- RISK SCORING ---
    risk_score = 50.0  # baseline
    
    # Factor: Contract value (larger deals = higher risk)
    contract_value = deal_data.get("contract_value", 0)
    if contract_value > 500000:
        risk_score += 20
    elif contract_value > 250000:
        risk_score += 10
    
    # Factor: Timeline tightness (fewer days = higher risk)
    days_to_close = deal_data.get("days_to_close", 30)
    if days_to_close < 7:
        risk_score += 25
    elif days_to_close < 14:
        risk_score += 15
    elif days_to_close < 30:
        risk_score += 5
    
    # Factor: Stakeholder unknowns or blockers
    blockers = deal_data.get("known_blockers", [])
    risk_score += len(blockers) * 10
    
    # Factor: Industry/segment risk (hardcoded categories)
    industry = deal_data.get("industry", "").lower()
    if "government" in industry or "defense" in industry:
        risk_score += 15
    
    risk_score = min(100, max(0, risk_score))
    
    # --- BUDGET FIT SCORING ---
    budget_fit = 50.0  # baseline
    
    proposed_cost = proposal_data.get("estimated_cost", 0)
    deal_budget = deal_data.get("budget", 1)  # avoid division by zero
    
    if deal_budget > 0:
        cost_ratio = proposed_cost / deal_budget
        if 0.8 <= cost_ratio <= 1.2:
            budget_fit = 95  # perfect fit
        elif 0.6 <= cost_ratio < 0.8:
            budget_fit = 80  # slightly under, good
        elif 1.2 < cost_ratio <= 1.5:
            budget_fit = 70  # slightly over, negotiable
        elif cost_ratio < 0.6 or cost_ratio > 1.5:
            budget_fit = 40  # way off
    
    # --- TIMELINE FEASIBILITY ---
    timeline_score = 50.0
    
    days_required = proposal_data.get("estimated_days", 60)
    days_available = deal_data.get("days_to_close", 30)
    
    if days_available >= days_required * 1.2:
        timeline_score = 95  # comfortable buffer
    elif days_available >= days_required:
        timeline_score = 80  # tight but doable
    elif days_available >= days_required * 0.8:
        timeline_score = 60  # risky, aggressive
    else:
        timeline_score = 30  # unrealistic
    
    # --- PROPOSAL QUALITY SCORING ---
    quality_score = 50.0
    
    # Completeness: number of required fields present
    required_fields = [
        "estimated_cost", "estimated_days", "success_metrics",
        "resource_allocation", "risk_mitigation"
    ]
    fields_present = sum(
        1 for field in required_fields if proposal_data.get(field)
    )
    quality_score = 40 + (fields_present / len(required_fields)) * 50
    
    # Clarity bonus: longer descriptions suggest more thought
    description_length = len(proposal_data.get("approach", ""))
    if description_length > 500:
        quality_score = min(100, quality_score + 15)
    elif description_length > 200:
        quality_score = min(100, quality_score + 5)
    
    quality_score = min(100, max(0, quality_score))
    
    # --- RECOMMENDATION LOGIC ---
    if risk_score > 75:
        recommendation = "reject"
        reason = "Risk score too high ({}). Deal has major blockers or timeline conflicts.".format(
            int(risk_score)
        )
    elif budget_fit < 50 or timeline_score < 40:
        recommendation = "revise"
        reason = "Proposal needs rework: budget misalignment or infeasible timeline."
    elif quality_score < 50:
        recommendation = "revise"
        reason = "Incomplete proposal. More detail required on resources and metrics."
    else:
        # All good; now consider composite
        composite = (100 - risk_score) * 0.3 + budget_fit * 0.2 + timeline_score * 0.2 + quality_score * 0.3
        if composite >= 70:
            recommendation = "approve"
            reason = "Strong overall fit. Deal is viable with manageable risk."
        else:
            recommendation = "revise"
            reason = "Marginal fit. Minor improvements needed to mitigate risks."
    
    logger.info(
        f"[Attempt {attempt}] {deal_id} deterministic: "
        f"risk={risk_score:.1f}, budget={budget_fit:.1f}, "
        f"timeline={timeline_score:.1f}, quality={quality_score:.1f} -> {recommendation}"
    )
    
    return ScoringResult(
        deal_id=deal_id,
        risk_score=risk_score,
        budget_fit_score=budget_fit,
        timeline_score=timeline_score,
        quality_score=quality_score,
        recommendation=recommendation,
        reasoning=reason,
        attempt=attempt,
        method="deterministic"
    )


# ============================================================================
# CLAUDE API SCORING
# ============================================================================

def claude_api_score(deal_data: Dict[str, Any],
                    proposal_data: Dict[str, Any],
                    attempt: int,
                    api_key: Optional[str] = None) -> Optional[ScoringResult]:
    """
    Use Claude API to score deal and proposal.
    Returns ScoringResult on success, None on failure (triggers fallback).
    
    Claude evaluates:
    - Deal risk (financial, timeline, stakeholder, market)
    - Budget fit (proposal cost vs deal budget)
    - Timeline feasibility (resource constraints, scope)
    - Proposal quality (thoroughness, alignment, clarity)
    - Final recommendation
    """
    
    deal_id = deal_data.get("deal_id", "UNKNOWN")
    logger.info(f"[Attempt {attempt}] Attempting Claude API scoring for {deal_id}")
    
    try:
        client = anthropic.Anthropic(api_key=api_key)
        
        prompt = f"""You are an expert sales operations evaluator. Score the following deal and proposal on multiple dimensions and provide a structured JSON response.

DEAL DATA:
{json.dumps(deal_data, indent=2)}

PROPOSAL DATA:
{json.dumps(proposal_data, indent=2)}

Evaluate and respond with ONLY valid JSON (no markdown, no preamble):
{{
  "risk_score": <0-100, higher=more risky>,
  "budget_fit_score": <0-100, higher=better fit>,
  "timeline_score": <0-100, higher=more feasible>,
  "quality_score": <0-100, higher=better quality>,
  "recommendation": "<approve|revise|reject>",
  "reasoning": "<brief explanation of recommendation>"
}}

Scoring guidelines:
- Risk: Consider deal size, timeline tightness, stakeholder complexity, known blockers, contract terms
- Budget fit: How well does proposal cost align with deal budget (target: ±10%)
- Timeline: Can scope realistically fit in days_to_close with quality?
- Quality: Is proposal complete, clear, and aligned with deal needs?
- Recommendation: approve (strong fit, manageable risk), revise (needs improvements), reject (unfeasible or too risky)
"""
        
        message = client.messages.create(
            model="claude-opus-4-6",
            max_tokens=500,
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        
        # Extract JSON from response
        response_text = message.content[0].text.strip()
        
        # Handle markdown code blocks if present
        if response_text.startswith("```"):
            response_text = response_text.split("```")[1]
            if response_text.startswith("json"):
                response_text = response_text[4:]
        response_text = response_text.strip()
        
        result_dict = json.loads(response_text)
        
        logger.info(
            f"[Attempt {attempt}] {deal_id} Claude API: "
            f"risk={result_dict.get('risk_score', -1):.1f}, "
            f"budget={result_dict.get('budget_fit_score', -1):.1f}, "
            f"timeline={result_dict.get('timeline_score', -1):.1f}, "
            f"quality={result_dict.get('quality_score', -1):.1f} -> {result_dict.get('recommendation')}"
        )
        
        return ScoringResult(
            deal_id=deal_id,
            risk_score=float(result_dict.get("risk_score", 50)),
            budget_fit_score=float(result_dict.get("budget_fit_score", 50)),
            timeline_score=float(result_dict.get("timeline_score", 50)),
            quality_score=float(result_dict.get("quality_score", 50)),
            recommendation=result_dict.get("recommendation", "revise"),
            reasoning=result_dict.get("reasoning", "Claude evaluation complete"),
            attempt=attempt,
            method="claude_api"
        )
    
    except anthropic.APIError as e:
        logger.warning(f"[Attempt {attempt}] Claude API error: {e}. Will retry or fall back.")
        return None
    except json.JSONDecodeError as e:
        logger.warning(f"[Attempt {attempt}] Failed to parse Claude response: {e}. Will retry or fall back.")
        return None
    except Exception as e:
        logger.warning(f"[Attempt {attempt}] Unexpected error during Claude API: {e}. Will retry or fall back.")
        return None


# ============================================================================
# EVALUATOR AGENT (Hybrid with Retries)
# ============================================================================

class EvaluatorAgent:
    """
    Multi-dimensional evaluator with hybrid Claude API + deterministic fallback.
    
    Workflow:
    1. Try Claude API (attempt 1)
    2. If fails, retry Claude (attempt 2)
    3. If fails again, try Claude one more time (attempt 3)
    4. If all Claude attempts fail, use deterministic fallback
    
    This demonstrates:
    - Tool use: Claude API calls
    - State management: Tracking attempt count, retry state
    - Conditional logic: Retry loop with fallback decision
    - Observability: Detailed logging at each step
    """
    
    def __init__(self, api_key: Optional[str] = None, max_retries: int = 3):
        self.api_key = api_key
        self.max_retries = max_retries
        self.evaluation_history = []  # Track all attempts for observability
        logger.info(f"EvaluatorAgent initialized (max_retries={max_retries}, "
                   f"claude_enabled={api_key is not None})")
    
    def evaluate(self,
                deal_data: Dict[str, Any],
                proposal_data: Dict[str, Any]) -> ScoringResult:
        """
        Evaluate deal and proposal using hybrid approach.
        
        Returns: ScoringResult with scores, recommendation, and metadata
        """
        deal_id = deal_data.get("deal_id", "UNKNOWN")
        logger.info(f"\n{'='*70}")
        logger.info(f"Starting evaluation for {deal_id}")
        logger.info(f"{'='*70}")
        
        # ===== ATTEMPT LOOP =====
        for attempt in range(1, self.max_retries + 1):
            logger.info(f"\nAttempt {attempt}/{self.max_retries}")
            
            # Try Claude API first
            if self.api_key:
                result = claude_api_score(deal_data, proposal_data, attempt, self.api_key)
                if result:
                    self.evaluation_history.append(result)
                    logger.info(f"SUCCESS on attempt {attempt} using Claude API")
                    logger.info(f"Final recommendation: {result.recommendation}")
                    return result
            
            # If Claude failed or not enabled, try again (log and continue)
            if attempt < self.max_retries:
                logger.info(f"Attempt {attempt} failed. Retrying...")
                continue
        
        # ===== FALLBACK =====
        logger.info(f"\nAll {self.max_retries} Claude attempts exhausted. Using deterministic fallback.")
        result = deterministic_score(deal_data, proposal_data, self.max_retries + 1)
        self.evaluation_history.append(result)
        logger.info(f"SUCCESS using deterministic fallback")
        logger.info(f"Final recommendation: {result.recommendation}")
        
        return result
    
    def get_history(self) -> list:
        """Return all evaluation attempts for audit/observability."""
        return [r.to_dict() for r in self.evaluation_history]


# ============================================================================
# INTEGRATION HELPER: Call from Orchestrator
# ============================================================================

def call_evaluator(orchestrator_state: Dict[str, Any],
                   api_key: Optional[str] = None) -> Dict[str, Any]:
    """
    Helper function to call Evaluator Agent from Orchestrator.
    
    Expects orchestrator_state to contain:
    - deal_data: Dict with deal metadata
    - proposal: Dict with proposal/draft
    
    Returns: orchestrator_state with evaluation_result added
    """
    logger.info("\n[ORCHESTRATOR] Calling Evaluator Agent")
    
    deal_data = orchestrator_state.get("deal_data", {})
    proposal = orchestrator_state.get("proposal", {})
    
    evaluator = EvaluatorAgent(api_key=api_key, max_retries=3)
    result = evaluator.evaluate(deal_data, proposal)
    
    orchestrator_state["evaluation_result"] = result.to_dict()
    orchestrator_state["evaluator_history"] = evaluator.get_history()
    
    logger.info(f"[ORCHESTRATOR] Evaluator complete: {result.recommendation}")
    
    return orchestrator_state


# ============================================================================
# EXAMPLE USAGE (for testing in Kaggle Notebook)
# ============================================================================

if __name__ == "__main__":
    
    # Mock deal and proposal
    mock_deal = {
        "deal_id": "DEAL-001",
        "account": "Acme Corp",
        "industry": "Technology",
        "contract_value": 450000,
        "budget": 500000,
        "days_to_close": 21,
        "known_blockers": ["CFO approval pending", "Technical integration concerns"]
    }
    
    mock_proposal = {
        "approach": "Implement enterprise platform with phased rollout. Phase 1: Core modules (14 days). Phase 2: Integrations (10 days). Includes 24/7 support and quarterly business reviews.",
        "estimated_cost": 480000,
        "estimated_days": 24,
        "resource_allocation": "5 engineers, 1 PM, 1 solutions architect",
        "risk_mitigation": "Weekly check-ins, escalation protocol for blockers, dedicated success manager",
        "success_metrics": "User adoption >80%, zero-downtime deployment, SLA >99.5%"
    }
    
    # Test without Claude API (deterministic only)
    print("\n" + "="*70)
    print("TEST 1: Deterministic Scoring Only")
    print("="*70)
    evaluator = EvaluatorAgent(api_key=None, max_retries=3)
    result = evaluator.evaluate(mock_deal, mock_proposal)
    print(json.dumps(result.to_dict(), indent=2))
    
    # Test with Claude API (if key available in environment)
    import os
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if api_key:
        print("\n" + "="*70)
        print("TEST 2: Hybrid Claude + Deterministic Fallback")
        print("="*70)
        evaluator_hybrid = EvaluatorAgent(api_key=api_key, max_retries=3)
        result_hybrid = evaluator_hybrid.evaluate(mock_deal, mock_proposal)
        print(json.dumps(result_hybrid.to_dict(), indent=2))
        print("\nEvaluation history:")
        print(json.dumps(evaluator_hybrid.get_history(), indent=2))

# Cell 3: Quick test
evaluator = EvaluatorAgent(api_key=None)  # Deterministic only
result = evaluator.evaluate(
    {"deal_id": "DEAL-001", "contract_value": 300000, "budget": 350000, "days_to_close": 45, "known_blockers": []},
    {"approach": "Good plan", "estimated_cost": 320000, "estimated_days": 35, "estimated_cost": 320000, "estimated_days": 35, "resource_allocation": "Team", "risk_mitigation": "Plan", "success_metrics": "KPIs"}
)
print(json.dumps(result.to_dict(), indent=2))
# Expected: recommendation="approve"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 23.9 MB/s eta 0:00:0000:01


2026-07-05 08:52:30,644 - [EvaluatorAgent] - INFO - EvaluatorAgent initialized (max_retries=3, claude_enabled=False)
2026-07-05 08:52:30,645 - [EvaluatorAgent] - INFO - 
2026-07-05 08:52:30,645 - [EvaluatorAgent] - INFO - Starting evaluation for DEAL-001
2026-07-05 08:52:30,646 - [EvaluatorAgent] - INFO - ======================================================================
2026-07-05 08:52:30,646 - [EvaluatorAgent] - INFO - 
Attempt 1/3
2026-07-05 08:52:30,647 - [EvaluatorAgent] - INFO - Attempt 1 failed. Retrying...
2026-07-05 08:52:30,648 - [EvaluatorAgent] - INFO - 
Attempt 2/3
2026-07-05 08:52:30,648 - [EvaluatorAgent] - INFO - Attempt 2 failed. Retrying...
2026-07-05 08:52:30,649 - [EvaluatorAgent] - INFO - 
Attempt 3/3
2026-07-05 08:52:30,650 - [EvaluatorAgent] - INFO - 
All 3 Claude attempts exhausted. Using deterministic fallback.
2026-07-05 08:52:30,650 - [EvaluatorAgent] - INFO - [Attempt 4] Using deterministic scoring for DEAL-001
2026-07-05 08:52:30,651 - [EvaluatorAgent]


TEST 1: Deterministic Scoring Only
{
  "deal_id": "DEAL-001",
  "risk_score": 85.0,
  "budget_fit_score": 95,
  "timeline_score": 60,
  "quality_score": 90.0,
  "composite_score": 62.5,
  "recommendation": "reject",
  "reasoning": "Risk score too high (85). Deal has major blockers or timeline conflicts.",
  "attempt": 4,
  "method": "deterministic",
  "timestamp": "2026-07-05T08:52:30.652043"
}
{
  "deal_id": "DEAL-001",
  "risk_score": 60.0,
  "budget_fit_score": 95,
  "timeline_score": 95,
  "quality_score": 90.0,
  "composite_score": 77.0,
  "recommendation": "approve",
  "reasoning": "Strong overall fit. Deal is viable with manageable risk.",
  "attempt": 4,
  "method": "deterministic",
  "timestamp": "2026-07-05T08:52:30.662785"
}
